# 06 - Comparaison SQL vs Neo4j

Ce notebook compare les approches relationnelles et graphes.

## Objectifs
- Montrer la difference de complexite des requetes
- Demontrer l'avantage de Neo4j pour les traversees

In [ ]:
import os
from neo4j import GraphDatabase
import time

In [ ]:
NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(os.environ.get('NEO4J_USER', 'neo4j'),
          os.environ['NEO4J_PASSWORD'])
)

def timed_query(cypher):
    start = time.time()
    with driver.session() as session:
        result = list(session.run(cypher))
    elapsed = (time.time() - start) * 1000
    return len(result), elapsed

## 1. Contact Tracing - Comparaison

### Version SQL (PostgreSQL)
```sql
WITH RECURSIVE contact_chain AS (
    SELECT patient_id, infected_by, 1 as level
    FROM infections 
    WHERE patient_id = 'P000000'
    
    UNION ALL
    
    SELECT i.patient_id, i.infected_by, cc.level + 1
    FROM infections i
    JOIN contact_chain cc ON i.infected_by = cc.patient_id
    WHERE cc.level < 3
)
SELECT * FROM contact_chain;
```

**Problemes SQL:**
- Requete recursive complexe
- Performance degradee avec la profondeur
- Difficile a maintenir

### Version Cypher (Neo4j)

In [ ]:
# Version Neo4j - simple et lisible
cypher = """
MATCH path = (p:Patient {patient_id: 'P000000'})-[:INFECTED_BY*1..3]->(contact)
RETURN contact.patient_id, length(path) AS profondeur
"""

count, elapsed = timed_query(cypher)
print(f"Resultats: {count} contacts")
print(f"Temps: {elapsed:.2f} ms")

## 2. Benchmark - Profondeur de traversee

Mesurons le temps pour differentes profondeurs.

In [ ]:
results = []
for depth in [1, 2, 3, 4, 5]:
    cypher = f"""
    MATCH path = (p:Patient {{patient_id: 'P000000'}})-[:INFECTED_BY*1..{depth}]->(c)
    RETURN count(DISTINCT c) AS contacts
    """
    count, elapsed = timed_query(cypher)
    results.append({'profondeur': depth, 'contacts': count, 'temps_ms': elapsed})
    print(f"Profondeur {depth}: {count} contacts en {elapsed:.2f} ms")

## 3. Avantages Neo4j

| Aspect | SQL | Neo4j |
|--------|-----|-------|
| **Syntaxe** | CTE recursive complexe | Pattern intuitif |
| **Performance** | O(n^depth) | O(edges) |
| **Lisibilite** | Faible | Excellente |
| **Maintenance** | Difficile | Facile |

### Cas d'usage ideaux pour Neo4j
- Contact tracing epidemiologique
- Analyse de reseaux sociaux
- Detection de fraude
- Recommandations
- Supply chain

In [ ]:
driver.close()